# 02 - Calibracion de Hiperparametros (k-Distance Graph), Metricas y HDBSCAN

## Curso de Machine Learning: Modelos No Supervisados

Aunque DBSCAN nos libera de tener que adivinar el numero de clusters $K$, transfiere la responsabilidad hacia dos hiperparametros geometricos que controlan que se considera "denso" en el espacio $\mathbb{R}^p$: el radio de vecindad **$\varepsilon$ (`eps`)** y el umbral de puntos minimos **$MinPts$ (`min_samples`)**.

Un error comun en la practica es probar valores de `eps` al azar hasta que el grafico "se vea bien". En espacios financieros de $p > 3$ dimensiones, la inspeccion visual directa es imposible y elegir `eps` arbitrariamente conduce a dos patologias opuestas:
1. **Fragmentacion y Exceso de Ruido ($\varepsilon$ muy pequeno)**: Las hiperesferas son tan diminutas que ningun punto alcanza `min_samples`, etiquetando el 90% del portafolio como anomalia (`-1`).
2. **Efecto Puente o Percolacion ($\varepsilon$ muy grande)**: Las hiperesferas son tan amplias que conectan regimenes distintos a traves de cadenas de puntos intermedios, colapsando todo el dataset en un unico cluster (`Cluster 0`).

En este cuaderno estudiaremos la metodologia formal basada en **Graficos de $k$-Distancias Ordenadas (`NearestNeighbors`)**, el calculo correcto de metricas de validacion en presencia de ruido y la evolucion hacia **HDBSCAN** para densidades variables.

---

### Contenidos
1. Anatomia de Hiperparametros: `min_samples` ($MinPts$) y Reglas Dimensionales
2. Determinacion Matematica de $\varepsilon$: El Grafico de $k$-Distancias Ordenadas (*k-Distance Elbow*)
3. El Impacto de la Metrica de Distancia (`metric`): Euclidiana ($L_2$), Manhattan ($L_1$) y Coseno
4. Evaluacion Correcta en DBSCAN: El Problema de la Etiqueta `-1` en Silhouette Score
5. Busqueda en Cuadricula Multicriterio (`eps` $\times$ `min_samples`)
6. La Limitacion de Densidad Variable y la Solucion Jerarquica: **HDBSCAN**
7. Comparativa Final en el Portafolio de Credito: DBSCAN Calibrado vs. HDBSCAN
8. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


In [ ]:
# Cargamos nuestro generador financiero del Cuaderno 01
def generar_dataset_credito_institucional(seed=42):
    rng = np.random.default_rng(seed)
    r0 = np.column_stack([
        rng.normal(1.4, 0.35, 220).clip(0.3, 2.5),
        rng.normal(11.5, 2.0, 220).clip(6.5, 18.0),
        45 + 18 * (rng.normal(1.4, 0.35, 220).clip(0.3, 2.5) ** 1.5) + rng.normal(0, 8, 220),
        rng.normal(16.0, 2.5, 220).clip(10.0, 23.0),
        rng.normal(4.2, 0.9, 220).clip(1.8, 7.5),
        rng.normal(14.5, 2.8, 220).clip(7.0, 22.0)
    ])
    r1 = np.column_stack([
        rng.normal(0.45, 0.25, 160).clip(0.0, 1.2),
        rng.normal(24.0, 3.5, 160).clip(15.0, 35.0),
        rng.normal(65.0, 12.0, 160).clip(35.0, 100.0),
        rng.normal(31.0, 3.8, 160).clip(22.0, 42.0),
        rng.normal(2.5, 0.6, 160).clip(1.0, 4.5),
        rng.normal(29.5, 4.2, 160).clip(20.0, 42.0)
    ])
    r2 = np.column_stack([
        rng.normal(4.3, 0.55, 140).clip(3.0, 5.8),
        rng.normal(2.8, 0.6, 140).clip(1.4, 4.5),
        260 + 25 * (rng.normal(4.3, 0.55, 140).clip(3.0, 5.8) ** 1.4) + rng.normal(0, 25, 140),
        rng.normal(42.0, 4.5, 140).clip(32.0, 54.0),
        rng.normal(14.5, 2.2, 140).clip(9.0, 21.0),
        rng.normal(6.5, 3.0, 140).clip(-2.0, 14.0)
    ])
    anomalias = np.column_stack([
        rng.uniform(7.5, 14.0, 30),
        rng.uniform(-3.5, 0.9, 30),
        rng.uniform(1100.0, 3800.0, 30),
        rng.uniform(70.0, 145.0, 30),
        rng.uniform(38.0, 95.0, 30),
        rng.uniform(-45.0, -12.0, 30)
    ])
    X_total = np.vstack([r0, r1, r2, anomalias])
    y_real = np.array([0]*220 + [1]*160 + [2]*140 + [-1]*30)
    perm = rng.permutation(len(X_total))
    columnas = [
        'deuda_neta_ebitda', 'cobertura_intereses_icr', 'spread_cds_5y_bps',
        'volatilidad_implicita_30d', 'bid_ask_spread_bps', 'retorno_roe_pct'
    ]
    df = pd.DataFrame(X_total[perm], columns=columnas)
    return df, y_real[perm]

df_credito, y_regimen_oculto = generar_dataset_credito_institucional()
scaler = RobustScaler()
X_rob = scaler.fit_transform(df_credito)


## 1. Seleccion del Primer Hiperparametro: `min_samples` ($MinPts$)

En la metodologia estandar de DBSCAN (Ester et al., 1996; Sander et al., 1998), **primero se fija `min_samples` basandose en la dimensionalidad $p$ y el tamano del dataset $n$**, y posteriormente se deduce $\varepsilon$ en funcion de `min_samples`.

### Reglas Heuristicas Fundamentales para `min_samples`:
1. **Cota Inferior Absoluta**: `min_samples` $\geq p + 1$, donde $p$ es el numero de variables (dimensiones).
   - *Nota critica*: Si `min_samples = 1`, cada punto aislado es un cluster en si mismo (0 ruido). Si `min_samples = 2`, DBSCAN se reduce matematicamente a un *Clustering Jerarquico de Enlace Simple (Single-Linkage)* truncado a la altura $\varepsilon$, sumamente propenso al efecto cadena.
2. **Regla de Sander para Datos con Ruido**: **`min_samples` $= 2 \cdot p$**.
   - En nuestro dataset de credito corporativo tenemos $p = 6$ variables financieras, por lo que el punto de partida teorico ideal es:
     $$MinPts = 2 \times 6 = 12 \quad (\text{o en el rango } [8, 15])$$

## 2. Determinacion Matematica de $\varepsilon$: El Grafico de $k$-Distancias Ordenadas

Una vez fijado $k = \text{min\_samples}$ (por ejemplo, $k = 10$ o $12$), ¿cual es el radio $\varepsilon$ exacto donde termina la densidad normal de los regimenes financieros y comienza el vacio de las anomalias?

### Algoritmo del Grafico de $k$-Distancias (*k-Distance Plot*):
1. Para cada observacion $x_i \in X$, calculamos la distancia hacia su **$k$-esimo vecino mas cercano** utilizando `sklearn.neighbors.NearestNeighbors(n_neighbors=k)`.
2. Ordenamos estas $n$ distancias de menor a mayor y las graficamos.
3. **Interpretacion de la Curva**:
   - La **meseta suave izquierda** corresponde a puntos situados dentro de regimenes densos (su $k$-esimo vecino esta muy cerca).
   - La **pared vertical derecha** corresponde a los *outliers* / anomalias (para encontrar $k$ vecinos necesitan expandir una esfera gigantesca a traves del espacio vacio).
   - El **punto de maxima curvatura (Rodilla / Knee)** marca el umbral critico $\varepsilon^*$ optimo.

In [ ]:
# Calculo de la curva de k-distancias con NearestNeighbors para k = 10k_vecinos = 10nn = NearestNeighbors(n_neighbors=k_vecinos, metric='euclidean')nn.fit(X_rob)distancias, indices = nn.kneighbors(X_rob)# Tomamos la columna k-1 (distancia al k-esimo vecino) y la ordenamos ascendentementek_dist_ordenadas = np.sort(distancias[:, k_vecinos - 1])# Deteccion matematica del punto de maxima curvatura (segunda derivada discreta / distancia a la cuerda)x_norm = np.linspace(0, 1, len(k_dist_ordenadas))y_norm = (k_dist_ordenadas - k_dist_ordenadas.min()) / (k_dist_ordenadas.max() - k_dist_ordenadas.min())# Distancia perpendicular desde cada punto de la curva a la linea recta que une los extremosvec_linea = np.array([1.0, 1.0]) / np.sqrt(2)puntos = np.column_stack([x_norm, y_norm])proyeccion = np.outer(np.dot(puntos, vec_linea), vec_linea)dist_perpendicular = np.linalg.norm(puntos - proyeccion, axis=1)idx_rodilla = np.argmax(dist_perpendicular)eps_optimo_knee = k_dist_ordenadas[idx_rodilla]plt.figure(figsize=(11, 5.5))plt.plot( k_dist_ordenadas, color='#1f77b4', linewidth=2.5, label=f'Distancia al {k_vecinos}-esimo vecino')plt.axhline(eps_optimo_knee, color='#d62728', linestyle='--', linewidth=2,             label=f'Rodilla Optima Detectada: eps = {eps_optimo_knee:.3f}')plt.axvline(idx_rodilla, color='gray', linestyle=':', alpha=0.8)plt.scatter([idx_rodilla], [eps_optimo_knee], color='#d62728', s=120, zorder=5)plt.annotate(f'Inicio de Zona de Ruido\n(eps* = {eps_optimo_knee:.2f}, Muestra #{idx_rodilla})',             xy=(idx_rodilla, eps_optimo_knee), xytext=(idx_rodilla - 140, eps_optimo_knee + 1.5),             arrowprops=dict(facecolor='black', shrink=0.08, width=1.5, headwidth=6),             fontsize=10, fontweight='bold', bbox=dict(boxstyle='round,pad=0.4', facecolor='wheat', alpha=0.9))plt.title(f'Metodo del Grafico de k-Distancias (k = min_samples = {k_vecinos})', fontsize=13, fontweight='bold')plt.xlabel('Emisiones Ordenadas por Densidad Decreciente (Indice)', fontsize=11, fontweight='bold')plt.ylabel(f'{k_vecinos}-Distancia Euclidiana (en Espacio RobustScaler)', fontsize=11, fontweight='bold')plt.legend(fontsize=11, loc='upper left')plt.tight_layout()plt.show()print(f"Radio optimo sugerido matematicamente por la rodilla de k-distancias: eps = {eps_optimo_knee:.4f}")

## 3. Evaluacion Correcta en DBSCAN: El Trampa de la Etiqueta `-1` en Silhouette

**Error critico frecuente**: Si pasas directamente `silhouette_score(X_rob, dbscan.labels_)` cuando existen anomalias (`-1`), Scikit-Learn interpretara que todos los puntos `-1` conforman **un cluster valido adicional**. Como las anomalias suelen estar dispersas por todos los extremos del espacio financiero, ese supuesto "cluster `-1`" tendra una dispersion gigantesca y hundira artificialmente el Silhouette Score.

### Regla de Oro para Evaluar DBSCAN:
1. **Filtrar la mascara de puntos agrupados (`mask = labels != -1`)** y calcular el **Silhouette Score Limpio** exclusivamente sobre los regimenes descubiertos (verificando que al menos existan $\geq 2$ clusters validos).
2. **Monitorear simultaneamente el Porcentaje de Ruido (`pct_ruido`)**: Si no controlamos la fraccion de ruido, un modelo podria obtener un Silhouette de $0.99$ dejando solo 10 puntos muy juntos y mandando el $98\%$ del dataset a `-1`.

In [ ]:
# Exploracion sistematica en cuadricula (eps x min_samples) con evaluacion limpia de ruidoeps_grid = [0.45, 0.65, 0.85, 1.10, 1.40]min_samples_grid = [6, 10, 14]resultados_grid = []for eps_val in eps_grid:    for ms_val in min_samples_grid:        db = DBSCAN(eps=eps_val, min_samples=ms_val, metric='euclidean')        lbls = db.fit_predict(X_rob)                mask_validos = (lbls != -1)        n_clusters = len(set(lbls[mask_validos]))        pct_ruido = (np.sum(~mask_validos) / len(lbls)) * 100                if n_clusters >= 2 and np.sum(mask_validos) > n_clusters:            sil_limpio = silhouette_score(X_rob[mask_validos], lbls[mask_validos])            db_index = davies_bouldin_score(X_rob[mask_validos], lbls[mask_validos])        else:            sil_limpio = np.nan            db_index = np.nan                    resultados_grid.append({            'eps': eps_val,            'min_samples': ms_val,            'N_Clusters': n_clusters,            'Pct_Ruido (%)': round(pct_ruido, 2),            'Silhouette_Limpio': round(sil_limpio, 4) if not np.isnan(sil_limpio) else None,            'Davies_Bouldin_Limpio': round(db_index, 4) if not np.isnan(db_index) else None,            'ARI_Real': round(adjusted_rand_score(y_regimen_oculto, lbls), 4)        })df_grid = pd.DataFrame(resultados_grid)print("Tabla de Calibracion Conjunta (eps x min_samples):")print(df_grid.to_string(index=False))

## 4. El Impacto de la Metrica de Distancia (`metric`): $L_2$ vs. $L_1$ (Manhattan)

En espacios de dimensionalidad moderada o alta ($p \geq 6$), la distancia Euclidiana ($L_2$) sufre una degradacion conocida como concentracion de distancias (la diferencia relativa entre el punto mas cercano y el mas lejano tiende a cero).

La **Distancia de Manhattan o Norma $L_1$** ($\|x_a - x_b\|_1 = \sum_{j=1}^p |x_{aj} - x_{bj}|$) no eleva al cuadrado las desviaciones individuales, ofreciendo mayor robustez frente a fluctuaciones puntuales en una sola variable financiera.

In [ ]:
# Comparacion de DBSCAN con metrica Manhattan (L1) vs Euclidiana (L2)db_l2 = DBSCAN(eps=0.85, min_samples=10, metric='euclidean').fit(X_rob)db_l1 = DBSCAN(eps=1.65, min_samples=10, metric='manhattan').fit(X_rob)print(f"Metrica Euclidiana (L2, eps=0.85): Clusters = {len(set(db_l2.labels_))-1} | Ruido = {(db_l2.labels_==-1).sum()} | ARI = {adjusted_rand_score(y_regimen_oculto, db_l2.labels_):.4f}")print(f"Metrica Manhattan  (L1, eps=1.65): Clusters = {len(set(db_l1.labels_))-1} | Ruido = {(db_l1.labels_==-1).sum()} | ARI = {adjusted_rand_score(y_regimen_oculto, db_l1.labels_):.4f}")

## 5. La Limitacion de Densidad Variable y la Evolucion a HDBSCAN

¿Cual es el talon de Aquiles de DBSCAN? **Utiliza un unico umbral global $\varepsilon$ para todo el espacio.**

En mercados financieros reales, el regimen de bonos *Investment Grade* suele ser sumamente compacto y denso (baja dispersion), mientras que el regimen de empresas *High-Yield* o *Biotecnologicas* es naturalmente mas disperso (mayor varianza intrinseca).

**HDBSCAN** (*Hierarchical DBSCAN*, Campello et al., 2013 — incluido nativamente en `sklearn.cluster.HDBSCAN` desde Scikit-Learn 1.3+) resuelve este problema integrando DBSCAN con el agrupamiento jerarquico:
1. Transforma el espacio definiendo la **Distancia de Alcance Mutuo (*Mutual Reachability Distance*)** para repeler puntos de baja densidad.
2. Construye el arbol generador minimo (*Minimum Spanning Tree*) y la jerarquia completa para **todos los valores posibles de $\varepsilon$ simultaneamente**.
3. Condensa el dendograma utilizando unicamente `min_cluster_size`, extrayendo los clusters mas persistentes a distintas escalas de densidad y proporcionando ademas una **probabilidad de pertenencia (`probabilities_`)** para cada activo.

In [ ]:
# Ajuste de HDBSCAN sin necesidad de especificar el radio epshdb = HDBSCAN(min_cluster_size=25, min_samples=10)labels_hdb = hdb.fit_predict(X_rob)probabilidades_hdb = hdb.probabilities_n_clust_hdb = len(set(labels_hdb[labels_hdb != -1]))ruido_hdb = (labels_hdb == -1).sum()ari_hdb = adjusted_rand_score(y_regimen_oculto, labels_hdb)print("=" * 70)print("RESULTADOS DE HDBSCAN (Adaptativo a Densidad Variable sin usar eps)")print("=" * 70)print(f"  Regimenes Descubiertos: {n_clust_hdb}")print(f"  Anomalias Detectadas:   {ruido_hdb} emisiones")print(f"  Pureza Global (ARI):    {ari_hdb:.4f}")print(f"  Confianza Media de Asignacion (Puntos no ruido): {probabilidades_hdb[labels_hdb != -1].mean():.4f}")# Visualizacion de la confianza probabilistica de HDBSCANplt.figure(figsize=(10, 5.5))sc = plt.scatter(df_credito['deuda_neta_ebitda'], df_credito['volatilidad_implicita_30d'],                 c=probabilidades_hdb, cmap='plasma', s=50, edgecolor='k', alpha=0.9)plt.colorbar(sc, label='Probabilidad de Pertenencia al Regimen (HDBSCAN)')plt.title('HDBSCAN: Mapa de Certeza de Pertenencia al Regimen (0.0 = Anomalia Pura, 1.0 = Nucleo Denso)',           fontsize=12, fontweight='bold')plt.xlabel('Deuda Neta / EBITDA (veces)', fontsize=11, fontweight='bold')plt.ylabel('Volatilidad Implicita 30d (%)', fontsize=11, fontweight='bold')plt.tight_layout()plt.show()

## 6. Conclusiones

### Puntos Clave:
1. **Orden de Calibracion**: Fija primero `min_samples` ($\approx 2 \cdot p$) y luego determina $\varepsilon$ localizando el codo de maxima curvatura en el **Grafico de $k$-Distancias Ordenadas**.
2. **Evaluacion Limpia de Ruido**: Calcula siempre `silhouette_score` y `davies_bouldin_score` filtrando previamente `labels != -1`, y acompanalo del reporte de porcentaje de ruido.
3. **HDBSCAN para Regimenes de Volatilidad Variable**: Cuando un portafolio combina sectores ultra-compactos (ej. *Utilities*) con sectores de alta dispersion (ej. *Venture / Biotech*), `HDBSCAN` elimina la necesidad de fijar $\varepsilon$ y entrega probabilidades de membresia.